©2026. For information, contact Deloitte Tohmatsu Group.

# 📝 演習概要

この演習では、RNN（リカレントニューラルネットワーク）をNumPy/CuPyでスクラッチ実装します。BPTT（Backpropagation Through Time）による時系列データの学習を、各層クラスや最適化手法を自作しながら構築し、RNNの内部動作と勾配伝播の仕組みを基礎から理解します。

# 事前準備

[JDLAが策定しているバージョン](https://www.jdla.org/certificate/engineer/)に合わせるために、以下のセルの実行をお願いします．

（#コメントアウト されているものは必要ありません）

また実行完了後に「ランタイムの再起動」をして下さい．

（以下のセルの実行は、最初にしていただければ、以降必要ありません．）


In [ ]:
# %%capture
# !pip uninstall matplotlib -y
# !pip install matplotlib==3.9.4

# !pip uninstall opencv-python -y
# !pip install opencv-python==4.11.0.86

# !pip uninstall torch -y
# !pip install torch==2.7.0

# !pip uninstall torchvision -y
# !pip install torchvision==0.22.0

## RNNにおけるBPTTの設定

## GPUの設定

In [ ]:
#GPUの設定
GPU = True
#GPU = False

if GPU:
    import cupy as np
    import cupyx
    np.cuda.set_allocator(np.cuda.MemoryPool().malloc)  # CuPyのメモリアロケータをメモリプールに差し替え

    print('\033[92m' + '-' * 60 + '\033[0m')
    print(' ' * 23 + '\033[92mGPU Mode (cupy)\033[0m')
    print('\033[92m' + '-' * 60 + '\033[0m\n')
else:
    import numpy as np

------------------------------------------------------------
                       GPU Mode (cupy)
------------------------------------------------------------



In [ ]:
def to_cpu(x):
    """
    cupy.ndarray -> numpy.ndarray に変換する。
    すでにCPU側(numpy.ndarray)ならそのまま返す。
    可視化・printするときはCPUメモリ上のndarrayのほうが便利なのでこれを使う。
    """
    import numpy
    if type(x) == numpy.ndarray:
        return x
    return np.asnumpy(x)  # cupy.ndarray → numpy.ndarray へコピー

def to_gpu(x):
    """
    numpy.ndarray -> cupy.ndarray に変換する。
    すでにGPU側(cupy.ndarray)ならそのまま返す。
    学習本体はGPU(CuPy)で回す前提なので、必要ならこれでGPUに移す。
    """
    import cupy
    import cupyx
    if type(x) == cupy.ndarray:
        return x
    return cupy.asarray(x)

## 層の設定

In [ ]:
# RNN層の設定
"""
RNN
・再帰ニューラルネットワーク
・コンテキストの単語の並び方を逃していることを対策するための層。
・コンテキストの長さによらず、そのコンテキストの情報を記憶するメカニズムをもつ。
・ループする経路（閉じた経路）をもつのが特徴で、そのことで過去の情報を記憶しながら、最新のデータへと更新できる。
"""

class RNN:
    def __init__(self, Wx, Wh, b):
        #引数でわたされたパラメタをメンバ変数paramsにリストとして設定。
        self.params = [Wx, Wh, b]
        #各パラメータに対応する形で勾配gradsを初期化し、格納
        self.grads = [np.zeros_like(Wx), np.zeros_like(Wh), np.zeros_like(b)]
        #逆伝播の計算時に使用する中間データcacheをNoneで初期化
        self.cache = None

    def forward(self, x, h_prev):
        """
        x: (N, D) 現在時刻の入力
        h_prev: (N, H) ひとつ前の隠れ状態
        return: h_next (N, H) 次の隠れ状態
        """
        Wx, Wh, b = self.params

        # RNN基本式: h_next = tanh(x Wx + h_prev Wh + b)
        t = np.dot(h_prev, Wh) + np.dot(x, Wx) + b
        h_next = np.tanh(t)

        # 逆伝播で使うので保持
        self.cache = (x, h_prev, h_next)
        return h_next

    def backward(self, dh_next):
        """
        dh_next: (N, H) 損失Lに対する「次の隠れ状態 h_next」の勾配 dL/dh_next
        return:
          dx      : (N, D) ひとつ前の層(埋め込みなど)への勾配
          dh_prev : (N, H) ひとつ前の時刻の隠れ状態への勾配
        勾配は self.grads にも格納する。
        """
        Wx, Wh, b = self.params
        x, h_prev, h_next = self.cache

        # tanhの微分: d/dt tanh(t) = 1 - tanh(t)^2
        dt = dh_next * (1 - h_next ** 2)

        # バイアス勾配: 時系列方向に和をとる（バッチ方向Nでsum）
        db = np.sum(dt, axis=0)

        # Wh勾配: h_prev^T dt
        dWh = np.dot(h_prev.T, dt)

        # dh_prev: dt Wh^T
        dh_prev = np.dot(dt, Wh.T)

        # Wx勾配: x^T dt
        dWx = np.dot(x.T, dt)

        # dx: dt Wx^T
        dx = np.dot(dt, Wx.T)

        # 勾配を保存（既存配列にブロードキャスト代入してるのはin-place更新）
        self.grads[0][...] = dWx
        self.grads[1][...] = dWh
        self.grads[2][...] = db

        return dx, dh_prev


Truncated BPTT（大きな時系列を扱うときに、時間軸方向に長くなりすぎたネットワークを適度に切り取ること）を実装するためにTimeごとに切り取る層にする。  
※BPTT：Backpropagation Through Time（時間方向に転換したニューラルネットワークの誤差逆伝播法）

In [ ]:
# TImeRNN層の設定
# T個(Tは任意の数)のRNNレイヤから構成される層
class TimeRNN:
    def __init__(self, Wx, Wh, b, stateful=False):#statefulはTimeRNN層の隠れ状態を維持するかしないかを意味する。長い時系列データを維持するときはTrueにする必要がある。
        # RNNセルが使う重みを共有するので、TimeRNNも同じparamsをまとめ保持
        self.params = [Wx, Wh, b]
        self.grads = [np.zeros_like(Wx), np.zeros_like(Wh), np.zeros_like(b)]
        self.layers = None  # 各時刻tごとにRNNセルを1個ずつ持っておく
        self.h, self.dh = None, None  # hは直前ステップの隠れ状態
        self.stateful = stateful

    def forward(self, xs):
        """
        xs: (N, T, D)
          N: バッチサイズ
          T: 時系列長
          D: 入力次元
        return:
          hs: (N, T, H) 各時刻の隠れ状態を全部並べたもの
        """
        Wx, Wh, b = self.params
        N, T, D = xs.shape
        D, H = Wx.shape

        self.layers = []
        hs = np.empty((N, T, H), dtype='f')#出力用の容器を用意する。

        # stateful=False か、もしくはまだhを持ってない場合はリセット
        if not self.stateful or self.h is None:
            self.h = np.zeros((N, H), dtype='f')


        # 時系列方向にRNNセルを順に適用していく
        for t in range(T):
            layer = RNN(*self.params)                       # 同じ重みで新しいRNNセルを生成
            self.h = layer.forward(xs[:, t, :], self.h)     # 1ステップ分forward
            hs[:, t, :] = self.h                            # 出力を保存
            self.layers.append(layer)                       # 逆伝播用に保持

        return hs

    def backward(self, dhs):
        """
        dhs: (N, T, H)
          各時刻の隠れ状態 h(t) に対する損失勾配 dL/dh(t)
        return:
          dxs: (N, T, D) 入力列xsに対する勾配
        また self.grads にパラメータの合計勾配を格納し、self.dh に「次バッチへ渡すべき h の勾配」を保持。
        """
        Wx, Wh, b = self.params
        N, T, H = dhs.shape
        D, H = Wx.shape

        dxs = np.empty((N, T, D), dtype='f')  # 各時刻のdxを格納する入れ物
        dh = 0                                # 次の時刻(将来)から伝わってくる勾配を蓄積
        grads = [0, 0, 0]                     # Wx, Wh, b の勾配を積算するための一時変数

        # 時刻T-1→0へ逆順にバックプロパゲーション（BPTT）
        for t in reversed(range(T)):
            layer = self.layers[t]
            # 将来からの勾配dhと、今回のdhs[:, t, :]を足してbackward
            dx, dh = layer.backward(dhs[:, t, :] + dh)#合算した勾配
            dxs[:, t, :] = dx

            # 各RNNセルの勾配を足しこんで合算
            for i, grad in enumerate(layer.grads):
                grads[i] += grad

        # 合算した勾配をメンバにコピーしておく
        for i, grad in enumerate(grads):
            self.grads[i][...] = grad

        # 最初の時刻に対する「前の隠れ状態h_prev」側の勾配を記憶
        self.dh = dh

        return dxs

    def set_state(self, h):
        """
        外から隠れ状態hを手動でセットする。（stateful RNNで続きをやりたいときに使う）
        """
        self.h = h

    def reset_state(self):
        """
        隠れ状態hを捨てる。stateful=Trueで学習していても明示的にリセットできる。
        """
        self.h = None


In [ ]:
class Embedding:
    """
    単語ID -> 埋め込みベクトル への変換を行う層。
    params[0] = W は shape (語彙数V, 埋め込み次元D)
    forwardで idx(=単語ID列) に対応する行を抜き出して返す。
    backwardでは対応する行に勾配を加算する（scatter_add / np.add.at）。
    """
    def __init__(self, W):
        self.params = [W]                     # 学習対象の単語埋め込み行列
        self.grads = [np.zeros_like(W)]       # その勾配
        self.idx = None                       # forwardで使った単語IDを記憶

    def forward(self, idx):
        """
        idx: (N,) などの整数ID配列
        return: (N, D) 対応する埋め込みベクトル
        """
        W, = self.params
        self.idx = idx
        out = W[idx]                          # IDに対応する行を取り出す（バッチ分）
        return out

    def backward(self, dout):
        """
        dout: (N, D) 埋め込みベクトルに対する勾配
        その勾配を対応する単語IDの行に足しこむ（=単語埋め込み行列への勾配）
        """
        dW, = self.grads
        dW[...] = 0  # 勾配をいったんゼロ初期化（同じIDが複数回出るので加算ベース）

        if GPU:
            # CuPy版のscatter_addでインデックスごとに勾配を加算
            cupyx.scatter_add(dW, self.idx, dout)
        else:
            # NumPy版: np.add.atで同じIDに対して勾配を加算
            np.add.at(dW, self.idx, dout)
        return None

In [ ]:
# TimeEmbeddingの設定

class TimeEmbedding:
    """
    時系列長TぶんのEmbeddingをまとめたレイヤ。
    xs: (N, T) の単語ID列を受け取り、
    各時刻tごとにEmbedding層で (N, D) に変換し、(N, T, D) を返す。
    backwardでは全時刻ぶんのEmbedding勾配を合計して1つの埋め込み行列勾配にする。
    """
    def __init__(self, W):
        self.params = [W]                     # 共有の埋め込み行列
        self.grads = [np.zeros_like(W)]       # 埋め込み行列への勾配
        self.layers = None                    # 各時刻tのEmbedding層インスタンス格納用
        self.W = W

    def forward(self, xs):
        """
        xs: (N, T) それぞれが単語ID
        return: (N, T, D) 各IDをEmbeddingした結果
        """
        N, T = xs.shape
        V, D = self.W.shape

        out = np.empty((N, T, D), dtype='f')
        self.layers = []

        # 各時刻tごとにEmbeddingを適用
        for t in range(T):
            layer = Embedding(self.W)         # 同じWを共有するEmbedding層
            out[:, t, :] = layer.forward(xs[:, t])
            self.layers.append(layer)

        return out

    def backward(self, dout):
        """
        dout: (N, T, D) TimeEmbedding出力への勾配
        各時刻tのEmbedding.backward()を呼び、勾配を足し合わせて self.grads[0] に格納。
        """
        N, T, D = dout.shape

        grad = 0
        for t in range(T):
            layer = self.layers[t]
            layer.backward(dout[:, t, :])     # この時刻t分の勾配をEmbeddingに反映
            grad += layer.grads[0]            # 各tでの勾配を足し込む

        self.grads[0][...] = grad
        return None


In [ ]:
class TimeAffine:
    """
    入力 x: (N, T, D)
    を、各タイムステップ独立に全結合 W,b で線形変換して
    出力: (N, T, V) を返す。
    """
    def __init__(self, W, b):
        self.params = [W, b]                 # W: (D, V), b: (V,)
        self.grads = [np.zeros_like(W), np.zeros_like(b)]
        self.x = None                        # forwardで使った入力を保存

    def forward(self, x):
        """
        x: (N, T, D)
        return: (N, T, V)
        """
        N, T, D = x.shape
        W, b = self.params

        # 時系列方向とバッチ方向をまとめて(N*T, D)にしたうえで一気に行列積
        rx = x.reshape(N*T, -1)
        out = np.dot(rx, W) + b

        self.x = x
        return out.reshape(N, T, -1)

    def backward(self, dout):
        """
        dout: (N, T, V) = dL/d(out)
        return: dx (N, T, D)
        さらに self.grads に dW, db を格納
        """
        x = self.x
        N, T, D = x.shape
        W, b = self.params

        # 同様に(N*T, V)にまとめてから逆伝播計算
        dout = dout.reshape(N*T, -1)
        rx = x.reshape(N*T, -1)

        db = np.sum(dout, axis=0)        # バイアス勾配
        dW = np.dot(rx.T, dout)          # 重み勾配
        dx = np.dot(dout, W.T)           # 入力側の勾配
        dx = dx.reshape(*x.shape)        # (N,T,D)に戻す

        self.grads[0][...] = dW
        self.grads[1][...] = db

        return dx

In [ ]:
# TimeSoftmaxWithLoss層
class TimeSoftmaxWithLoss:
    """
    RNNの出力 (N,T,V) と 教師ラベル (N,T) を受けて、
    時間方向すべてのクロスエントロピー損失の平均を返す。
    ignore_label==-1 のときは、そのタイムステップを損失計算に含めない(マスクする)。
    """
    def __init__(self):
        self.params, self.grads = [], []  # 学習パラメータはなし
        self.cache = None
        self.ignore_label = -1

    def forward(self, xs, ts):
        """
        xs: (N,T,V) RNNの出力 (未softmax)
        ts: (N,T)   正解ラベルID もしくは (N,T,V) one-hot
        return: スカラー損失
        """
        N, T, V = xs.shape

        # 教師ラベルが one-hot の場合は argmax でIDに戻す
        if ts.ndim == 3:
            ts = ts.argmax(axis=2)

        # ignore_label で無視する位置をマスク
        mask = (ts != self.ignore_label)

        # バッチと時間をまとめて (N*T, V), (N*T,)
        xs = xs.reshape(N * T, V)
        ts = ts.reshape(N * T)
        mask = mask.reshape(N * T)

        # softmax計算
        ys = softmax(xs)  # (N*T, V)

        # 正解ラベルtsに対応する確率をとってlogをとる
        ls = np.log(ys[np.arange(N * T), ts])
        ls *= mask  # マスクがFalseの場所は0にして無効化
        loss = -np.sum(ls)
        loss /= mask.sum()  # 有効なステップ数で割って平均化

        # 逆伝播で使う一式を保存
        self.cache = (ts, ys, mask, (N, T, V))
        return loss

    def backward(self, dout=1):
        """
        dout: 上流からのスカラー勾配(基本1)
        return: dx (N,T,V) = dL/dxs
        """
        ts, ys, mask, (N, T, V) = self.cache

        dx = ys
        # 正解クラスのところから1を引く = softmax+CEの基本的な勾配
        dx[np.arange(N * T), ts] -= 1

        dx *= dout
        dx /= mask.sum()  # 有効ステップ数で割る
        dx *= mask[:, np.newaxis]  # ignore_label部分は0にする

        dx = dx.reshape((N, T, V))
        return dx

In [ ]:
#活性関数
def sigmoid(x):
    """シグモイド関数"""
    return 1 / (1 + np.exp(-x))


def relu(x):
    """ReLU関数"""
    return np.maximum(0, x)


def softmax(x):
    """
    ソフトマックス関数
    - オーバーフロー防止のため、最大値を引いてからexpする
    - xが2次元(N,V)ならバッチごとにsoftmax
    - xが1次元(V,)ならその1つにsoftmax
    戻り値のshapeは x と同じ
    """
    if x.ndim == 2:
        x = x - x.max(axis=-1, keepdims=True)
        x = np.exp(x)
        x /= x.sum(axis=-1, keepdims=True)
    elif x.ndim == 1:
        x = x - np.max(x)
        x = np.exp(x) / np.sum(np.exp(x))
    return x

## 最適化手法の設定

In [ ]:
#  最適化手法の設定
#  確率的勾配降下法（Stochastic Gradient Descent）
class SGD:
    """
    params[i] -= lr * grads[i]
    を繰り返してパラメータを更新する。
    """
    def __init__(self, lr=0.01):
        self.lr = lr

    def update(self, params, grads):
        # 渡された params と grads のリストを同じインデックスで対応させて更新
        for i in range(len(params)):
            params[i] -= self.lr * grads[i]

## データセットの獲得

In [ ]:
# データセットの獲得
import os
try:
    import urllib.request
except ImportError:
    raise ImportError('Use Python3!')
import pickle
# import numpy as np


url_base = 'https://raw.githubusercontent.com/tomsercu/lstm/master/data/'
key_file = {
    'train':'ptb.train.txt',
    'test':'ptb.test.txt',
    'valid':'ptb.valid.txt'
}
save_file = {
    'train':'ptb.train.npy',
    'test':'ptb.test.npy',
    'valid':'ptb.valid.npy'
}
vocab_file = 'ptb.vocab.pkl'

dataset_dir = os.getcwd()
# os.path.dirname(os.path.abspath(__file__))


def _download(file_name):
    """
    指定した file_name がローカルに無ければ URL からダウンロードして保存する。
    """
    file_path = dataset_dir + '/' + file_name
    if os.path.exists(file_path):
        return

    print('Downloading ' + file_name + ' ... ')

    try:
        urllib.request.urlretrieve(url_base + file_name, file_path)
    except urllib.error.URLError:
        # SSLエラー対策で検証なしコンテキストを使うフォールバック
        import ssl
        ssl._create_default_https_context = ssl._create_unverified_context
        urllib.request.urlretrieve(url_base + file_name, file_path)

    print('Done')


def load_vocab():
    """
    語彙辞書(word_to_id, id_to_word)を作る/読み込む。
    - すでにpickleがあるならそれを読む
    - なければ train テキストを読んで作る
    """
    vocab_path = dataset_dir + '/' + vocab_file

    # 既に保存済みならそれをロード
    if os.path.exists(vocab_path):
        with open(vocab_path, 'rb') as f:
            word_to_id, id_to_word = pickle.load(f)
        return word_to_id, id_to_word

    # なければ新規に作る
    word_to_id = {}
    id_to_word = {}
    data_type = 'train'
    file_name = key_file[data_type]
    file_path = dataset_dir + '/' + file_name

    _download(file_name)

    # PTBコーパスは改行ごとに行が区切られているので <eos> を挟んで1列化
    words = open(file_path).read().replace('\n', '<eos>').strip().split()

    # 各単語にIDを割り振る
    for i, word in enumerate(words):
        if word not in word_to_id:
            tmp_id = len(word_to_id)
            word_to_id[word] = tmp_id
            id_to_word[tmp_id] = word

    # pickleで保存
    with open(vocab_path, 'wb') as f:
        pickle.dump((word_to_id, id_to_word), f)

    return word_to_id, id_to_word


def load_data(data_type='train'):
    '''
        :param data_type: データの種類：'train' or 'test' or 'valid (val)'
        :return:
    '''
    if data_type == 'val': data_type = 'valid'
    save_path = dataset_dir + '/' + save_file[data_type]

    word_to_id, id_to_word = load_vocab()

    # すでにnpyがあるなら読み込んで即return
    if os.path.exists(save_path):
        corpus = np.load(save_path)
        return corpus, word_to_id, id_to_word

    # なければテキストをダウンロード→ID列に変換→保存
    file_name = key_file[data_type]
    file_path = dataset_dir + '/' + file_name
    _download(file_name)

    words = open(file_path).read().replace('\n', '<eos>').strip().split()

    # 各単語をIDに変換
    corpus = np.array([word_to_id[w] for w in words])

    # .npyに保存（次回以降速く読むため）
    np.save(save_path, corpus)

    return corpus, word_to_id, id_to_word


# スクリプトとして直接実行された場合は train/val/test を全部前処理して保存する
if __name__ == '__main__':
    for data_type in ('train', 'val', 'test'):
        load_data(data_type)

Done
Done
Done


モデルの設定

In [ ]:
#モデルSimpleRnnlmの設定

class SimpleRnnlm:
    def __init__(self, vocab_size, wordvec_size, hidden_size):
        V, D, H = vocab_size, wordvec_size, hidden_size
        rn = np.random.randn  # 正規分布乱数ショートカット

        # 重みの初期化
        #  Embedding用: 単語ID→単語ベクトル (V×D)
        embed_W = (rn(V, D) / 100).astype('f')

        # RNN用: 入力→隠れ, 隠れ→隠れ, バイアス
        rnn_Wx = (rn(D, H) / np.sqrt(D).astype('f'))
        rnn_Wh = (rn(H, H) / np.sqrt(H).astype('f'))
        rnn_b = np.zeros(H).astype('f')

        # 出力用Affine: 隠れ→語彙スコア (H×V) とバイアス (V,)
        affine_W = (rn(H, V) / np.sqrt(H)).astype('f')
        affine_b = np.zeros(V).astype('f')

        #レイヤの生成
        self.layers = [
            TimeEmbedding(embed_W),
            TimeRNN(rnn_Wx, rnn_Wh, rnn_b, stateful=True),  # stateful=Trueなのでバッチ間でhを引き継ぐ設計
            TimeAffine(affine_W, affine_b)
        ]
        # Softmax+Loss (時系列版)
        self.loss_layer = TimeSoftmaxWithLoss()
        # RNN層だけ直接触れるように保持（stateリセット用など）
        self.rnn_layer = self.layers[1]

        # すべてのパラメータと勾配を1つのリストにまとめる
        self.params, self.grads = [], []
        for layer in self.layers:
            self.params += layer.params
            self.grads += layer.grads

    def forward(self, xs, ts):
        """
        xs: (N,T) 入力単語ID
        ts: (N,T) 正解となる次単語ID
        return: 損失スカラー
        """
        # Embedding → RNN → Affine を順にforward
        for layer in self.layers:
            xs = layer.forward(xs)

        # 出てきた (N,T,V) と正解 (N,T) から時系列Softmax+CE損失を計算
        loss = self.loss_layer.forward(xs, ts)
        return loss

    def backward(self, dout=1):
        """
        TimeSoftmaxWithLoss から逆向きに戻っていって
        各層の勾配を self.grads に反映する。
        """
        dout = self.loss_layer.backward(dout)
        for layer in reversed(self.layers):
            dout = layer.backward(dout)
        return dout

    def reset_state(self):
        """stateful RNN の隠れ状態 h をリセットしたいときに呼ぶ"""
        self.rnn_layer.reset_state()


モデルの学習

In [ ]:
import matplotlib.pyplot as plt
# import numpy as np

# ハイパーパラメータ設定
batch_size = 10      # ミニバッチサイズ
wordvec_size = 100   # 単語ベクトル次元 D
hidden_size = 100    # RNN隠れ状態次元 H
time_size = 5        # BPTTで一度に展開する時間長(T)
lr = 0.1             # 学習率
max_epoch = 500      # 学習エポック数

# PTBデータの読み込み（train）
corpus, word_to_id, id_to_word = load_data('train')

# データを小さく制限（1000語ぶんだけ使う）
corpus_size = 1000
corpus = corpus[:corpus_size]

# 語彙サイズ（IDの最大+1）
vocab_size = int(max(corpus) + 1)

# 言語モデルの学習では、
# 入力 xs(t) から次の単語 ts(t)=xs(t+1) を予測させたいので1単語ずらす
xs = corpus[:-1]  # 入力系列 (長さ corpus_size-1)
ts = corpus[1:]   # 正解系列 (1語先)

data_size = len(xs)
print('corpus size: %d, vocabulary size: %d' % (corpus_size, vocab_size))

# イテレーション関連の初期化
max_iters = data_size // (batch_size * time_size)  # 1エポックあたりの反復回数
time_idx = 0
total_loss = 0
loss_count = 0
ppl_list = []  # perplexity(困惑度)の推移を保存

# モデルとオプティマイザの用意
model = SimpleRnnlm(vocab_size, wordvec_size, hidden_size)
optimizer = SGD(lr)

# ① 各バッチサンプルがどこから読み始めるか、オフセットを決める
#    例: バッチサイズ10なら、コーパスを10分割してそれぞれ別の開始点から読むイメージ
jump = (corpus_size - 1) // batch_size
offsets = [i * jump for i in range(batch_size)]

# 学習ループ
for epoch in range(max_epoch):
    for iter in range(max_iters):
        # ② ミニバッチを組み立てる
        #    batch_x: (batch_size, time_size)
        #    batch_t: (batch_size, time_size)
        batch_x = np.empty((batch_size, time_size), dtype='i')
        batch_t = np.empty((batch_size, time_size), dtype='i')

        for t in range(time_size):
            for i, offset in enumerate(offsets):
                # offsets[i] から time_idx だけ進んだ位置の単語IDをとる
                batch_x[i, t] = xs[(offset + time_idx) % data_size]
                batch_t[i, t] = ts[(offset + time_idx) % data_size]
                time_idx += 1  # 時系列を進める

        # ③ forward/backward/パラメータ更新
        loss = model.forward(batch_x, batch_t)  # 損失計算
        model.backward()                        # 勾配計算
        optimizer.update(model.params, model.grads)  # SGDでパラメータ更新

        total_loss += loss
        loss_count += 1

    # ④ 1エポックごとに PPL(perplexity) を計算・表示
    #    perplexity = exp(平均損失) -- 言語モデルの指標としてよく使う
    ppl = np.exp(total_loss / loss_count)
    print('| epoch %d | perplexity %.2f' % (epoch+1, ppl))

    ppl_list.append(float(ppl))
    total_loss, loss_count = 0, 0  # エポックごとにリセット

corpus size: 1000, vocabulary size: 418
| epoch 1 | perplexity 302.81
| epoch 2 | perplexity 166.64
| epoch 3 | perplexity 172.42
| epoch 4 | perplexity 207.02
| epoch 5 | perplexity 194.16
| epoch 6 | perplexity 165.24
| epoch 7 | perplexity 161.56
| epoch 8 | perplexity 193.20
| epoch 9 | perplexity 220.41
| epoch 10 | perplexity 187.89
| epoch 11 | perplexity 161.74
| epoch 12 | perplexity 137.58
| epoch 13 | perplexity 139.66
| epoch 14 | perplexity 168.64
| epoch 15 | perplexity 180.19
| epoch 16 | perplexity 156.41
| epoch 17 | perplexity 144.41
| epoch 18 | perplexity 140.89
| epoch 19 | perplexity 173.92
| epoch 20 | perplexity 177.48
| epoch 21 | perplexity 169.49
| epoch 22 | perplexity 117.07
| epoch 23 | perplexity 122.13
| epoch 24 | perplexity 121.36
| epoch 25 | perplexity 147.30
| epoch 26 | perplexity 154.68
| epoch 27 | perplexity 125.50
| epoch 28 | perplexity 123.82
| epoch 29 | perplexity 135.82
| epoch 30 | perplexity 152.93
| epoch 31 | perplexity 149.17
| epoch 

学習経過のプロット

In [ ]:
import matplotlib.pyplot as plt

epochs = range(max_epoch)
plt.plot(epochs, ppl_list, 'b')
plt.title('Perplexity')       # 困惑度の推移
plt.xlabel('Epochs')          # 横軸: エポック
plt.ylabel('Perplexity')      # 縦軸: PPL(小さいほど良い)
plt.show()

## 🔧 実践問題1：Truncated BPTTの展開長を変えてパープレキシティの変化を観察する

上のコードでは `time_size=5` でBPTTを展開しています。
これはRNNが一度に「5単語先まで」の依存関係しか学習できないことを意味します。

| パラメータ | 大きくすると | 小さくすると |
|:---|:---|:---|
| `time_size` | より長い文脈を学習できるが、勾配消失が起きやすくなり計算コストも増す | 計算は速いが、長距離の依存関係を捉えられない |
| `hidden_size` | モデルの記憶容量が増えるが、過学習しやすくなる | 計算が軽いが、表現力が不足する場合がある |

**問題：** 以下のコードの `______` を自分で決めて、`time_size` と `hidden_size` を変更した場合のパープレキシティの変化を比較してください。

> **注意1：** `time_size` を変えると1エポックあたりのイテレーション数（`max_iters`）も変わります。下のコードではそれを自動計算しています。

> **注意2：** `max_epoch` を大きくしすぎると時間がかかります。100〜300程度がおすすめです。


In [ ]:
# BPTT展開長と隠れ層サイズを変えて実験する

# TODO: 自分で値を設定してください
time_size2 = ______     # BPTT展開長
hidden_size2 = ______   # 隠れ状態の次元数（元は100）
wordvec_size2 = 100     # 単語ベクトル次元
max_epoch2 = ______     # エポック数（元は500）
lr2 = 0.1
batch_size2 = 10

# データ再読み込み（元と同じ1000語）
corpus2, word_to_id2, id_to_word2 = load_data('train')
corpus2 = corpus2[:1000]
vocab_size2 = int(max(corpus2) + 1)
xs2 = corpus2[:-1]
ts2 = corpus2[1:]
data_size2 = len(xs2)

# time_sizeが変わるとmax_itersも変わる
max_iters2 = data_size2 // (batch_size2 * time_size2)
print(f'time_size={time_size2}, max_iters/epoch={max_iters2}')

# モデル作成
model2 = SimpleRnnlm(vocab_size2, wordvec_size2, hidden_size2)
optimizer2 = SGD(lr2)

jump2 = (1000 - 1) // batch_size2
offsets2 = [i * jump2 for i in range(batch_size2)]
time_idx2 = 0
total_loss2 = 0
loss_count2 = 0
ppl_list2 = []

for epoch in range(max_epoch2):
    for iter in range(max_iters2):
        batch_x = np.empty((batch_size2, time_size2), dtype='i')
        batch_t = np.empty((batch_size2, time_size2), dtype='i')
        for t in range(time_size2):
            for i, offset in enumerate(offsets2):
                batch_x[i, t] = xs2[(offset + time_idx2) % data_size2]
                batch_t[i, t] = ts2[(offset + time_idx2) % data_size2]
                time_idx2 += 1
        loss = model2.forward(batch_x, batch_t)
        model2.backward()
        optimizer2.update(model2.params, model2.grads)
        total_loss2 += loss
        loss_count2 += 1

    ppl = np.exp(total_loss2 / loss_count2)
    if (epoch+1) % 50 == 0:
        print(f'| epoch {epoch+1} | perplexity {ppl:.2f}')
    ppl_list2.append(float(ppl))
    total_loss2, loss_count2 = 0, 0

# 元の結果と比較プロット
plt.plot(ppl_list[:len(ppl_list2)], label=f'original (T=5, H=100)', alpha=0.7)
plt.plot(ppl_list2, label=f'new (T={time_size2}, H={hidden_size2})', alpha=0.7)
plt.xlabel('Epochs'); plt.ylabel('Perplexity')
plt.legend(); plt.title('BPTT time_size comparison'); plt.show()


<details><summary>解答例</summary>

```python
# 例1: 展開長を伸ばす（より長い文脈を学習）
time_size2 = 15
hidden_size2 = 100
max_epoch2 = 200

# 例2: 展開長と隠れ層の両方を変える
time_size2 = 10
hidden_size2 = 150
max_epoch2 = 300
```

- `time_size=15` にすると、RNNが一度に15単語分の依存関係を学習できるため、パープレキシティが改善する傾向があります
- ただし `time_size` が大きすぎると勾配消失が深刻になり、逆に学習が不安定になることがあります（これがLSTM/GRUが生まれた理由です）
- `time_size` を変えると `max_iters` が変わる点に注意してください。`max_iters = data_size // (batch_size × time_size)` なので、`time_size` を大きくすると1エポックあたりのイテレーション数が減ります
- 元のモデル（T=5）とパープレキシティの推移を比較し、収束速度と最終値の両方に注目してください
</details>


## 🔧 実践問題2：勾配クリッピングを実装して学習を安定させる

RNNの学習では、BPTTで時間方向に勾配を伝播するため**勾配爆発**（gradient explosion）が起きやすくなります。
勾配爆発が起きると損失がNaNになったり、パープレキシティが突然跳ね上がったりします。

これを防ぐ手法が**勾配クリッピング**（gradient clipping）です：

1. 全パラメータの勾配のL2ノルム（大きさ）を計算する
2. ノルムが閾値（`max_norm`）を超えていたら、閾値に収まるよう勾配を縮小する

数式で書くと：
```
L2ノルム = sqrt( Σ ||grad||² )    ← 全勾配の二乗和の平方根
もし L2ノルム > max_norm ならば:
    各勾配 *= (max_norm / L2ノルム)   ← 比率で縮小
```

---

**問題：** 以下の `clip_grads` 関数の `______` を埋めて勾配クリッピングを実装し、学習に適用してください。

<br/>

<details>
<summary>💡 <b>ヒント（クリックして表示）</b></summary>

<blockquote>
<code>np.sum(grad ** 2)</code> で1つの勾配配列の二乗和が計算できます。全パラメータについてこれを合計してから平方根を取ります。
</blockquote>

</details>

<br/>


In [ ]:
def clip_grads(grads, max_norm):
    """勾配クリッピング: 全勾配のL2ノルムがmax_normを超えたら縮小する"""
    # 全パラメータの勾配の二乗和を計算
    total_norm = 0
    for grad in grads:
        total_norm += np.______(grad ** ______)
    total_norm = np.______(total_norm)  # 平方根を取ってL2ノルムにする

    # ノルムが閾値を超えていたら、比率で縮小する
    rate = max_norm / (total_norm + 1e-6)  # ゼロ除算防止
    if rate ______ 1:
        for grad in grads:
            grad ______ rate  # 各勾配をrateで縮小


# 勾配クリッピング付きで学習を再実行
model3 = SimpleRnnlm(vocab_size, wordvec_size, hidden_size)
optimizer3 = SGD(0.1)
max_norm = 5.0  # クリッピングの閾値

jump3 = (1000 - 1) // 10
offsets3 = [i * jump3 for i in range(10)]
time_idx3 = 0
total_loss3, loss_count3 = 0, 0
ppl_list3 = []

for epoch in range(300):
    for iter in range(max_iters):
        batch_x = np.empty((10, 5), dtype='i')
        batch_t = np.empty((10, 5), dtype='i')
        for t in range(5):
            for i, offset in enumerate(offsets3):
                batch_x[i, t] = xs[(offset + time_idx3) % data_size]
                batch_t[i, t] = ts[(offset + time_idx3) % data_size]
                time_idx3 += 1
        loss = model3.forward(batch_x, batch_t)
        model3.backward()

        # 勾配クリッピングを適用してからパラメータ更新
        clip_grads(model3.grads, max_norm)
        optimizer3.update(model3.params, model3.grads)

        total_loss3 += loss
        loss_count3 += 1

    ppl = np.exp(total_loss3 / loss_count3)
    if (epoch+1) % 50 == 0:
        print(f'| epoch {epoch+1} | perplexity {ppl:.2f}')
    ppl_list3.append(float(ppl))
    total_loss3, loss_count3 = 0, 0

plt.plot(ppl_list[:300], label='without clipping', alpha=0.7)
plt.plot(ppl_list3, label=f'with clipping (max_norm={max_norm})', alpha=0.7)
plt.xlabel('Epochs'); plt.ylabel('Perplexity')
plt.legend(); plt.title('Gradient Clipping effect'); plt.show()


<details><summary>解答例</summary>

```python
def clip_grads(grads, max_norm):
    total_norm = 0
    for grad in grads:
        total_norm += np.sum(grad ** 2)
    total_norm = np.sqrt(total_norm)

    rate = max_norm / (total_norm + 1e-6)
    if rate < 1:
        for grad in grads:
            grad *= rate
```

- `np.sum(grad ** 2)` で各勾配配列の二乗和を計算し、全パラメータについて合計してから `np.sqrt` でL2ノルムを求めます
- `rate < 1` のとき（＝ノルムが閾値を超えているとき）だけ縮小を適用します。`rate >= 1` なら勾配はそのままです
- `grad *= rate` はin-place演算で、元の勾配配列を直接書き換えます。これにより `model.grads` の中身も更新されます
- 勾配クリッピングは特にRNN/LSTM/GRUで重要で、BPTTで時間方向に勾配が掛け合わされて爆発するのを防ぎます
- `max_norm` の値は5.0〜10.0程度が一般的です。小さすぎると学習が遅くなり、大きすぎるとクリッピングの効果が薄れます
</details>
